# Harness ranking notebook — the T2 reference rankings (H2.3)

Brute-force system-level ranking of the adsorbent database: every
honestly-flagged material from the H1.0 D–A fit export plus the curated
anchors, through `Cycle0D-v0` per application profile, with the top
candidates refined through the dynamic `Bed1D-v0`.

This is the **reference ranking** the adsorbent-ml Stage-2 surrogate's
top-k hit rate is scored against — the model proposes, this table
disposes. The sweep set excludes flagged fits, unphysical `q_sat`, and
single-temperature isotherms (no `Q_st`) up front; the excluded counts
print below so the denominator is always visible.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)

from harness.materials import load_anchors
from harness.rank import (
    load_sweep_materials,
    refine_with_bed1d,
    shortlist,
    sweep_materials,
)

mats = load_sweep_materials() + load_anchors()
ranked = sweep_materials(mats)
print(f"{len(mats)} materials x {ranked['profile'].nunique()} profiles")

[harness.rank] /home/samu2505/ENTERPRISE/Cooling-with-heat/data_cache/fits/da_params.csv: 1221 fits → 553 loadable → 145 usable rows, 21 adsorbents (dropped: 245 flagged, 20 unphysical q_sat)


34 materials x 4 profiles


In [2]:
from pathlib import Path

out_dir = Path("data_cache/rankings")
out_dir.mkdir(parents=True, exist_ok=True)
ranked.to_csv(out_dir / "cycle0d_sweep.csv", index=False)

for profile in ("datacenter", "cpu", "human", "vehicle"):
    print(f"== {profile} top-10 (score = profile-weighted normalized COP/SCP) ==")
    display(shortlist(ranked, profile, 10)[
        ["rank", "material", "source", "COP", "SCP_W_kg", "delta_q", "score"]])

== datacenter top-10 (score = profile-weighted normalized COP/SCP) ==


,rank,material,source,COP,SCP_W_kg,delta_q,score
128,1,MIL-101(Cr),anchor,1.715114,2542.863143,0.310273,1.000000
110,2,MIL-100,isodb,0.567250,522.875379,0.063800,0.495045
134,3,Activated carbon (BPL-type),anchor,0.535762,293.471672,0.035809,0.406840
123,4,Silica gel RD,anchor,0.484761,314.895983,0.038423,0.378771
107,5,CuBTC,isodb,0.446340,311.036684,0.037952,0.351783
132,6,Aluminum fumarate,anchor,0.427320,300.586211,0.036677,0.335928
118,7,Xtrusorb oxidized,isodb,0.491288,141.282641,0.017239,0.332449
113,8,Silica Gel,isodb,0.436059,215.996926,0.026355,0.317101
129,9,UiO-66,anchor,0.409524,267.187743,0.032601,0.314194
114,10,Sorbonorit 4 Activated Carbon,isodb,0.449362,138.284074,0.016873,0.303354


== cpu top-10 (score = profile-weighted normalized COP/SCP) ==


,rank,material,source,COP,SCP_W_kg,delta_q,score
26,1,MIL-101(Cr),anchor,1.777994,14852.734683,0.726220,1.000000
9,2,MIL-160,isodb,0.688745,2351.370693,0.114970,0.937979
32,3,Activated carbon (BPL-type),anchor,0.672552,2926.261060,0.143079,0.931751
8,4,MIL-100,isodb,0.643204,4662.464242,0.227970,0.920463
5,5,CuBTC,isodb,0.626122,3913.000583,0.191325,0.913893
21,6,Silica gel RD,anchor,0.607334,3344.120044,0.163510,0.906667
30,7,Aluminum fumarate,anchor,0.573215,3599.878614,0.176015,0.893544
11,8,Silica Gel,isodb,0.557402,2121.082394,0.103710,0.887462
27,9,UiO-66,anchor,0.550910,3199.892102,0.156458,0.884966
29,10,CAU-10-H,anchor,0.532195,2449.567215,0.119771,0.877767


== human top-10 (score = profile-weighted normalized COP/SCP) ==


,rank,material,source,COP,SCP_W_kg,delta_q,score
60,1,MIL-101(Cr),anchor,1.296530,437.301803,0.106149,0.754705
43,2,MIL-160,isodb,0.759261,694.473945,0.168573,0.733345
39,3,CuBTC,isodb,0.601434,622.881690,0.151195,0.586718
42,4,MIL-100,isodb,0.591982,437.405666,0.106174,0.539712
66,5,Activated carbon (BPL-type),anchor,0.611589,346.286891,0.084056,0.536828
64,6,Aluminum fumarate,anchor,0.547661,545.711454,0.132464,0.525627
55,7,Silica gel RD,anchor,0.569287,433.792439,0.105297,0.520037
61,8,UiO-66,anchor,0.525278,485.076848,0.117745,0.494182
63,9,CAU-10-H,anchor,0.504002,382.877882,0.092938,0.454891
46,10,Sorbonorit 4 Activated Carbon,isodb,0.540705,168.741888,0.040960,0.440301


== vehicle top-10 (score = profile-weighted normalized COP/SCP) ==


,rank,material,source,COP,SCP_W_kg,delta_q,score
77,1,MIL-160,isodb,0.766740,3075.169711,0.223347,0.965308
73,2,CuBTC,isodb,0.571671,2155.214881,0.156531,0.884030
98,3,Aluminum fumarate,anchor,0.511712,1774.641336,0.128890,0.859046
95,4,UiO-66,anchor,0.488205,1577.458965,0.114569,0.839741
78,5,MOF-74-Ni,isodb,0.420145,2079.605911,0.151040,0.820894
101,6,LiCl/silica SWS-1L,anchor,0.438077,1526.001852,0.110832,0.797143
97,7,CAU-10-H,anchor,0.473622,1390.950714,0.101023,0.754970
96,8,Mg-MOF-74 (CPO-27-Mg),anchor,0.431359,1431.345647,0.103957,0.754404
92,9,AlPO-18,anchor,0.447325,1341.886544,0.097460,0.723310
93,10,SAPO-34,anchor,0.426478,1241.797093,0.090190,0.672392


## Anchor positions per profile

The anchors are the calibration set for these rankings: where they land
must match the known screening results (zeolites cannot regenerate at
60 °C; water-adsorbing MOFs and carbons lead the low-grade-heat cases).

In [3]:
anchors = ranked[ranked["source"] == "anchor"]
display(anchors.pivot_table(index="material", columns="profile", values="rank"))

profile,cpu,datacenter,human,vehicle
material,,,,
Activated carbon (BPL-type),3.0,3.0,5.0,15.0
AlPO-18,12.0,19.0,12.0,9.0
Aluminum fumarate,7.0,6.0,6.0,3.0
CAU-10-H,10.0,14.0,9.0,7.0
LiCl/silica SWS-1L,15.0,21.0,15.0,6.0
MIL-101(Cr),1.0,1.0,1.0,34.0
Mg-MOF-74 (CPO-27-Mg),14.0,20.0,14.0,8.0
SAPO-34,17.0,22.0,17.0,10.0
Silica gel RD,6.0,4.0,7.0,11.0


## The regeneration-temperature story: zeolite 13X vs silica gel RD

13X binds water strongly (E_char = 14 kJ/mol, Q_st = 3.5 MJ/kg), so a
60 °C regeneration against a 35 °C condenser barely desorbs it, while
its high `Q_st` inflates the heat input. Warm the regeneration to
75–80 °C and the gap closes — the ranking is
regeneration-temperature-driven, which is the system-level signal the
surrogate has to learn.

In [4]:
sel = ranked[ranked["material"].isin(["Zeolite 13X (NaX)", "Silica gel RD"])]
display(sel.pivot_table(index="material", columns="profile",
                        values=["COP", "delta_q"]).round(4))

COP                            delta_q                           
profile               cpu datacenter   human vehicle     cpu datacenter   human vehicle
material                                                                               
Silica gel RD      0.6073     0.4848  0.5693  0.4500  0.1635     0.0384  0.1053  0.0606
Zeolite 13X (NaX)  0.3107     0.1554  0.2877  0.3143  0.0360     0.0064  0.0324  0.0623

## Dynamic refinement — datacenter top-5 through `Bed1D-v0`

The equilibrium sweep carries no transport assumptions; the dynamic
refinement does. Fitted rows get class-default `k_ldf`/`k_eff`
(`transport_provenance="default"`): orderings under the fixed transport
assumption are meaningful, absolute SCP across materials is not (§8.1
honesty note).

In [5]:
refined = refine_with_bed1d(ranked, "datacenter", k=5)
display(refined.round(4))

,profile,material,rank_cycle0d,COP,SCP_W_kg,delta_q,COP_cycle0d,SCP_cycle0d,transport_provenance
0,datacenter,MIL-101(Cr),1,1.3538,764.1936,0.0932,1.7151,2542.8631,default
1,datacenter,MIL-100,2,0.5805,266.5551,0.0325,0.5672,522.8754,default
2,datacenter,Activated carbon (BPL-type),3,0.6005,229.2476,0.0280,0.5358,293.4717,default
3,datacenter,Silica gel RD,4,0.5229,220.0464,0.0268,0.4848,314.8960,default
4,datacenter,CuBTC,5,0.4520,214.6888,0.0262,0.4463,311.0367,default


## What drives the datacenter ranking?

In [6]:
dc = ranked[ranked["profile"] == "datacenter"].copy()
print("score correlations (datacenter):")
display(dc[["COP", "SCP_W_kg", "delta_q", "q_sat_kg_kg", "Q_st_MJ_kg",
            "e_char_j_mol", "score"]].corr()["score"].round(3))
print("out-of-window rows (isotherm window does not cover the setpoints):",
      int(dc["out_of_window"].sum()), "of", len(dc))
display(dc.nsmallest(5, "rank")[
    ["rank", "material", "delta_q", "Q_st_MJ_kg", "e_char_j_mol", "out_of_window"]])

score correlations (datacenter):


COP             0.976
SCP_W_kg        0.862
delta_q         0.862
q_sat_kg_kg     0.759
Q_st_MJ_kg     -0.586
e_char_j_mol   -0.484
score           1.000
Name: score, dtype: float64

out-of-window rows (isotherm window does not cover the setpoints): 9 of 34


,rank,material,delta_q,Q_st_MJ_kg,e_char_j_mol,out_of_window
128,1,MIL-101(Cr),0.310273,0.800000,3000.000000,False
110,2,MIL-100,0.063800,2.471742,3691.808048,False
134,3,Activated carbon (BPL-type),0.035809,2.200000,4000.000000,False
123,4,Silica gel RD,0.038423,2.500000,4500.000000,False
107,5,CuBTC,0.037952,2.332908,6299.311877,False


## Findings (H2.3)

- The datacenter shortlist is led by the water-adsorbing MOFs and
  carbons; **zeolite 13X lands in the bottom third** — the known
  screening result, reproduced by the harness (gate:
  `tests/harness/test_h2_3_shortlist_sanity.py`).
- The ranking is regeneration-temperature-driven: 13X/silica COP ratio
  improves ~1.6× from the 60 °C (datacenter) to the 80 °C (human)
  regeneration.
- On the datacenter 45–70 °C loop the band bottom is degenerate for
  strong-binding materials (zero swing below ≈ 60 °C regeneration) —
  the same marginality that makes the H2.2 schedule question a
  non-issue there.
- Dynamic refinement reorders the top of the shortlist under transients;
  transport defaults flag it (`transport_provenance="default"`) —
  closing that gap with per-material `k_eff` is the natural H3 head.

Artifacts: `data_cache/rankings/cycle0d_sweep.csv` (working copy,
gitignored); the committed reference is this notebook's outputs.